Download the dataset and Extract it



In [ ]:
!wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz


--2023-08-01 10:57:43--  https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘aclImdb_v1.tar.gz’

aclImdb_v1.tar.gz   100%[===================>]  80.23M  17.5MB/s    in 8.1s    

2023-08-01 10:57:52 (9.91 MB/s) - ‘aclImdb_v1.tar.gz’ saved [84125825/84125825]



Install Transformers Library

In [ ]:
!pip install transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 38.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 25.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 109.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.2 MB/s eta 0:00:00


Import Essential Libraries

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer, RobertaModel
from torch.optim import AdamW
from sklearn.metrics import classification_report


Path to the dataset

In [ ]:
path = "aclImdb"

Initialize the Roberta tokenizer

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

ImdbDataset class to preprocess and load samples from the IMDB dataset

In [ ]:
class ImdbDataset(Dataset):
    def __init__(self, reviews, targets, tokenizer, max_len):
      """
        Initialize the ImdbDataset.

        Parameters:
        - reviews (list): A list containing movie review texts.
        - targets (list): A list containing sentiment labels for each review.
        - tokenizer (Tokenizer): Tokenizer object to tokenize the reviews.
        - max_len (int): Maximum length for tokenized sequences.
        """
        self.reviews = reviews
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
      """
        Returns:
        int: Number of samples in the dataset.
        """
        return len(self.reviews)

    def __getitem__(self, item):
       """
        Retrieve a single sample from the dataset.

        Parameters:
        - item (int): Index for the desired sample.

        Returns:
        dict: A dictionary containing tokenized review data and the sentiment label.
        """
        review = str(self.reviews[item])
        target = self.targets[item]

        encoding = self.tokenizer.encode_plus(
            review,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'review_text': review,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(target, dtype=torch.long)
        }

    Create and return a DataLoader object for the given data and targets

In [ ]:
def create_data_loader(data, targets, tokenizer, max_len, batch_size):
    dataset = ImdbDataset(
        reviews=data,
        targets=targets,
        tokenizer=tokenizer,
        max_len=max_len
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=4
    )

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:560: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Read the IMDB data from the given directory

In [ ]:
def read_imdb_split(split_dir):
    split_dir = os.path.join(path, split_dir)
    texts = []
    labels = []
    for label_dir in ["pos", "neg"]:
        dir_name = os.path.join(split_dir, label_dir)
        for fname in glob.glob(os.path.join(dir_name, "*.txt")):
            with open(fname) as f:
                texts.append(f.read())
            labels.append(0 if label_dir == "neg" else 1)

    return texts, labels

Read in the IMDB train and test datasets

In [ ]:
train_texts, train_labels = read_imdb_split('train')
test_texts, test_labels = read_imdb_split('test')

Create data loaders for training and testing

In [ ]:
train_data_loader = create_data_loader(train_texts, train_labels, tokenizer, max_len=256, batch_size=3)
test_data_loader = create_data_loader(test_texts, test_labels, tokenizer, max_len=256, batch_size=3)

a custom model class that uses RoBERTa with an additional adapter layer for sentiment classification

In [ ]:
class RobertaWithAdapter(nn.Module):
    def __init__(self, num_labels=2):
        super(RobertaWithAdapter, self).__init__()
        self.num_labels = num_labels
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.adapter = nn.Linear(self.roberta.config.hidden_size, 64)  # Add an adapter layer
        self.classifier = nn.Linear(64, self.num_labels)# Classification layer

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        # Use the hidden state of first token ([CLS]) for classification
        cls_token_hidden_state = outputs[0][:, 0, :]
        adapter_out = F.relu(self.adapter(cls_token_hidden_state))  # Apply F.relu() before passing the output to the classifier
        logits = self.classifier(adapter_out)
        return logits

Check if GPU is available and set the device accordingly

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Initialize the model, optimizer, and loss function

In [ ]:
model = RobertaWithAdapter().to(device)
optimizer = AdamW(model.parameters(), lr=1e-5)
loss_fn = nn.CrossEntropyLoss().to(device)

Define function for training the model



In [ ]:
def train_model(model, data_loader, loss_fn, optimizer, device, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0
    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["targets"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # print(f"Outputs shape: {outputs.shape}")
        # print(f"Targets shape: {targets.shape}")

        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, targets)
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

    return correct_predictions.double() / n_examples, np.mean(losses)

Define function for evaluating the model.

In [ ]:
def eval_model(model, data_loader, loss_fn, device, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0
    predictions = []
    real_values = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["targets"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, targets)
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())

            predictions.extend(preds)
            real_values.extend(targets)

    predictions = torch.stack(predictions).cpu()
    real_values = torch.stack(real_values).cpu()
    classification_rep = classification_report(real_values, predictions, output_dict=True)

    return correct_predictions.double() / n_examples, np.mean(losses), classification_rep


Train the model for several epochs and evaluate its performance on the test data

In [ ]:
EPOCHS = 5
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)
    train_acc, train_loss = train_model(
        model,
        train_data_loader,
        loss_fn,
        optimizer,
        device,
        len(train_texts)
    )
    print(f'Train loss {train_loss} accuracy {train_acc}')
    val_acc, val_loss, val_classification_rep = eval_model(
        model,
        test_data_loader,
        loss_fn,
        device,
        len(test_texts)
    )
    print(f'Val   loss {val_loss} accuracy {val_acc}')
    print(f"Val   precision {val_classification_rep['macro avg']['precision']}")
    print(f"Val   recall {val_classification_rep['macro avg']['recall']}")
    print(f"Val   F1-score {val_classification_rep['macro avg']['f1-score']}")
    print()


Epoch 1/5
----------
Train loss 0.005018397663971307 accuracy 0.9993600000000001
Val   loss 6.859424875527229 accuracy 0.5
Val   precision 0.25
Val   recall 0.5
Val   F1-score 0.3333333333333333

Epoch 2/5
----------


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloa

Train loss 0.01054292059497923 accuracy 0.9985200000000001
Val   loss 7.329448358429517 accuracy 0.5
Val   precision 0.25
Val   recall 0.5
Val   F1-score 0.3333333333333333

Epoch 3/5
----------


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloa

Train loss 0.007164374160326872 accuracy 0.9988000000000001
Val   loss 7.695245093483396 accuracy 0.5
Val   precision 0.25
Val   recall 0.5
Val   F1-score 0.3333333333333333

Epoch 4/5
----------


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloa

Train loss 0.009844081266374833 accuracy 0.9986
Val   loss 7.83494360084467 accuracy 0.5
Val   precision 0.25
Val   recall 0.5
Val   F1-score 0.3333333333333333

Epoch 5/5
----------


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloa

Train loss 0.010303621828441889 accuracy 0.99792
Val   loss 7.740618293308806 accuracy 0.5
Val   precision 0.25
Val   recall 0.5
Val   F1-score 0.3333333333333333



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
